In [4]:
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
from skimage.feature import hog, local_binary_pattern
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load Data
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
# Subset for speed: 10k for training, 2k for testing
X_tr_raw, y_tr = train_images[:10000], train_labels[:10000]
X_te_raw, y_te = test_images[:2000], test_labels[:2000]

def extract_features(images, mode="combined"):
    features_list = []
    for img in images:
        # --- HOG Component ---
        # Captures the shape/edges
        fd_hog = hog(img, orientations=9, pixels_per_cell=(7, 7), 
                     cells_per_block=(2, 2), visualize=False)
        
        # --- Spatial LBP Component ---
        # Divide image into 4x4 blocks to keep spatial info
        radius = 1
        n_points = 8 * radius
        lbp = local_binary_pattern(img, n_points, radius, method='uniform')
        
        spatial_lbp = []
        cell_size = 7 # 28 / 7 = 4 cells per axis
        for r in range(0, 28, cell_size):
            for c in range(0, 28, cell_size):
                cell = lbp[r:r+cell_size, c:c+cell_size]
                hist, _ = np.histogram(cell, bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
                spatial_lbp.extend(hist.astype("float") / (hist.sum() + 1e-7))
        
        fd_lbp = np.array(spatial_lbp)

        # Selection logic
        if mode == "hog":
            features_list.append(fd_hog)
        elif mode == "lbp":
            features_list.append(fd_lbp)
        else: # combined
            features_list.append(np.hstack([fd_hog, fd_lbp]))
            
    return np.array(features_list)

# 2. Extract all three sets
print("Extracting features...")
X_train_hog = extract_features(X_tr_raw, "hog")
X_test_hog = extract_features(X_te_raw, "hog")

X_train_lbp = extract_features(X_tr_raw, "lbp")
X_test_lbp = extract_features(X_te_raw, "lbp")

X_train_comb = extract_features(X_tr_raw, "combined")
X_test_comb = extract_features(X_te_raw, "combined")

# 3. Training & Evaluation Helper
def evaluate(X_train, X_test, y_train, y_test, name):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    clf = LinearSVC(max_iter=5000, random_state=42, dual=False)
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"{name} Accuracy: {acc*100:.2f}%")

evaluate(X_train_hog, X_test_hog, y_tr, y_te, "HOG Only")
evaluate(X_train_lbp, X_test_lbp, y_tr, y_te, "Spatial LBP Only")
evaluate(X_train_comb, X_test_comb, y_tr, y_te, "HOG + LBP Combined")

Extracting features...
HOG Only Accuracy: 81.90%
Spatial LBP Only Accuracy: 77.55%
HOG + LBP Combined Accuracy: 84.05%
